In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
compare_cloud_masks.py

Comparaison entre :
    - le masque nuage natif Landsat
    - le masque bayésien

Définition des masques :
    1    = ciel clair
    0    = nuage
    NaN  = terre

Comparaison :
    0    = concordance
    +1   = natif clair / bayésien nuage
    -1   = natif nuage / bayésien clair
    NaN  = terre ou absence de comparaison

Produits :
    1. Masque natif
    2. Masque bayésien
    3. Différence des masques
    4. BT11 avec masque natif
    5. BT11 avec masque bayésien
    6. Statistiques de comparaison
Version 3, k Correa
"""

import os
import warnings

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree
from matplotlib.colors import ListedColormap, BoundaryNorm

warnings.filterwarnings("ignore", category=RuntimeWarning)

# 1) PARAMETRES
# Landsat
landsat_f0 = '20240118111136-IFR-L1C_GHRSST-BT-LANDSAT9-CALISTA_v1.1-v02.2-fv1.0.nc'
#landsat_f0 = '20230103104903-IFR-L1C_GHRSST-BT-LANDSAT9-CALISTA_v1.1-v02.2-fv1.0.nc'
#landsat_f0 = '20221216110012-IFR-L1C_GHRSST-BT-LANDSAT9-CALISTA_v1.1-v02.2-fv1.0.nc'
#landsat_f0 = '20221020110600-IFR-L1C_GHRSST-BT-LANDSAT9-CALISTA_v1.1-v02.2-fv1.0.nc'

#landsat_f0 = 'CALISTA_LC09_L1C_202027_20221216_20230317_02_T1.nc'
#landsat_f0 = 'CALISTA_LC09_L1C_200030_20230103_20230315_02_T1.nc'
#landsat_f0 = 'CALISTA_LC09_L1C_203026_20221020_20230325_02_T1.nc'

landsat_file = os.path.join("/home/kcorrea/Trishna/data/l1c/l1c_v2/", landsat_f0)
#landsat_file = os.path.join("/home/kcorrea/Trishna/data/l1c/", landsat_f0)


# Bayésien 
cfname = 'CALISTA_LC09_L1C_204025_20240118_20240118_0_x00.095_k40_w3_bayesian_pixels_ajustPara2f2.nc'
#cfname = 'CALISTA_LC09_L1C_200030_20230103_20230315_0_x00.095_k40_w3_bayesian_pixels_ajustPara2f2.nc'
#cfname = 'CALISTA_LC09_L1C_202027_20221216_20230317_0_x00.095_k40_w3_bayesian_pixels_ajustPara2f2.nc'
#cfname = 'CALISTA_LC09_L1C_203026_20221020_20230325_0_x00.095_k40_w3_bayesian_pixels_ajustPara2f2.nc'
#cfname = 'CALISTA_LC09_L1C_203026_20221020_20230325_0_x00.095_k40_w3_bayesian_pixels_ajustPara2f2.nc'

file_bayes = os.path.join("/home/kcorrea/Trishna/bayesian/nc_outputfile/", cfname)

#  Sorties

outdir = "/home/kcorrea/Trishna/figures/new3/"
os.makedirs(outdir, exist_ok=True)

# Seuil bayésien 

threshold = 0.3

# 2) LECTURE DES DONNEES

print("Lecture Landsat...")
ds = xr.open_dataset(landsat_file)

print("Lecture Bayesian...")
ds_bayes = xr.open_dataset(file_bayes)

# 3) DONNEES LANDSAT

BT11 = (ds["brightness_temperature_11"].squeeze().values.astype(np.float32))

lat = ds["lat"].squeeze().values
lon = ds["lon"].squeeze().values

nj = ds.sizes["nj"]
ni = ds.sizes["ni"]

valid_BT = np.isfinite(BT11)

print("\n======================================")
print("GRILLE LANDSAT")
print("======================================")
print("nj =", nj)
print("ni =", ni)
print("BT11 =", BT11.shape)

# 4) MASQUE TERRE

# validity_flags :
#     0 = océan
#     1 = terre

validity_flags = (ds["validity_flags"].squeeze().values)

terre = validity_flags == 1
ocean = validity_flags == 0

# Terre à superposer sur les figures
# 1 = terre
# NaN = océan
land_hold_on = np.where( terre, 1.0, np.nan)

print("\n======================================")
print("MASQUE TERRE")
print("======================================")
print("Océan :", np.sum(ocean))
print("Terre :", np.sum(terre))

# 5) MASQUE NUAGE NATIF LANDSAT

cfmask = ds["cloud_mask"]

cloud_mask_values = (cfmask.values.astype(np.int64))

# cloud_mask :
#
# bit 0 = masque nuage natif Landsat
# bit 1 = masque MSG
#
# Ici on utilise uniquement le bit 0.

mask_cloud_native = ( (cloud_mask_values & 1) != 0)

mask_cloud_native = np.squeeze(mask_cloud_native)


print("\n======================================")
print("MASQUE NUAGE NATIF")
print("======================================")
print("Nuages :", np.sum(mask_cloud_native))
print("Clairs :", np.sum(~mask_cloud_native))

# 6) CONSTRUCTION DU MASQUE NATIF FINAL

# Définition :
#
#     1    = ciel clair
#     0    = nuage
#     NaN  = terre / BT11 invalide

mask_native = np.full( BT11.shape, np.nan, dtype=np.float32)

# Eau + ciel clair
# ----------------------------------------------------------

mask_native[ocean & (~mask_cloud_native) & valid_BT] = 1.0

# Eau + nuage
# ----------------------------------------------------------

mask_native[ocean & mask_cloud_native & valid_BT] = 0.0


print("\n======================================")
print("MASQUE NATIF FINAL")
print("======================================")
print("Clear :", np.sum(mask_native == 1))
print("Cloud :", np.sum(mask_native == 0))
print("NaN   :", np.sum(np.isnan(mask_native)))

# 7) DONNEES BAYESIENNES

row = ds_bayes["row"].values
col = ds_bayes["col"].values

Pc = ds_bayes["Pc2"].values

TB11_bayes = (
    ds_bayes["TB10_obs"]
    .values
)

lat_bayes = ds_bayes["lat"].values
lon_bayes = ds_bayes["lon"].values


print("\n======================================")
print("PRODUIT BAYESIEN")
print("======================================")
print("Nombre de pixels :", len(Pc))
print("row :", row.min(), row.max())
print("col :", col.min(), col.max())

# 8) VALIDATION ROW / COL

valid = (
    np.isfinite(row)
    & np.isfinite(col)
    & (row >= 0)
    & (row < nj)
    & (col >= 0)
    & (col < ni)
)

print(
    "\nPixels row/col valides :",
    np.sum(valid),
    "/",
    len(valid)
)


row_v = row[valid].astype(np.int64)
col_v = col[valid].astype(np.int64)

Pc_v = Pc[valid]
TB11_v = TB11_bayes[valid]

lat_b_v = lat_bayes[valid]
lon_b_v = lon_bayes[valid]

# 9) VERIFICATION ROW/COL PAR LAT/LON

lat_test = lat[row_v, col_v]
lon_test = lon[row_v, col_v]

distance_deg = np.sqrt(
    (lat_test - lat_b_v) ** 2
    +
    (lon_test - lon_b_v) ** 2
)

valid_geo = (
    np.isfinite(distance_deg)
    &
    np.isfinite(lat_b_v)
    &
    np.isfinite(lon_b_v)
)


geo_threshold = 1e-5

if np.any(valid_geo):

    print("\n======================================")
    print("CORRESPONDANCE ROW/COL")
    print("======================================")

    print(
        "Distance moyenne :",
        np.nanmean(distance_deg[valid_geo])
    )

    print(
        "Distance médiane :",
        np.nanmedian(distance_deg[valid_geo])
    )

    print(
        "Distance maximale :",
        np.nanmax(distance_deg[valid_geo])
    )

    same_grid = (
        np.nanmedian(distance_deg[valid_geo])
        < geo_threshold
    )

else:

    same_grid = False


print("\nMême grille ?", same_grid)


# 10) RECONSTRUCTION DU PRODUIT BAYESIEN

Pc_2D = np.full( (nj, ni), np.nan, dtype=np.float32)

TB11_bayes_2D = np.full( (nj, ni), np.nan, dtype=np.float32)

# 10A) ROW/COL DIRECTS

if same_grid:

    print("\n>>> Utilisation directe de row/col")

    Pc_2D[
        row_v,
        col_v
    ] = Pc_v

    TB11_bayes_2D[
        row_v,
        col_v
    ] = TB11_v


# 10B) CORRESPONDANCE LAT/LON

else:

    print("\n>>> Correspondance par latitude/longitude")

    lat_flat = lat.ravel()
    lon_flat = lon.ravel()

    valid_ls = (
        np.isfinite(lat_flat)
        &
        np.isfinite(lon_flat)
    )

    coords_ls = np.column_stack([
        lat_flat[valid_ls],
        lon_flat[valid_ls]
    ])

    print("Construction du KDTree...")

    tree = cKDTree(coords_ls)

    coords_bayes = np.column_stack([
        lat_b_v,
        lon_b_v
    ])

    distance, index = tree.query(
        coords_bayes,
        k=1
    )

    max_distance = 1e-4

    valid_match = (
        np.isfinite(distance)
        &
        (distance < max_distance)
    )

    print(
        "Pixels bayésiens associés :",
        np.sum(valid_match),
        "/",
        len(valid_match)
    )

    original_indices = np.where(
        valid_ls
    )[0]

    flat_indices = original_indices[
        index[valid_match]
    ]

    row_match, col_match = np.unravel_index(
        flat_indices,
        (nj, ni)
    )

    Pc_2D[
        row_match,
        col_match
    ] = Pc_v[valid_match]

    TB11_bayes_2D[
        row_match,
        col_match
    ] = TB11_v[valid_match]


# 11) MASQUE BAYESIEN

# Définition :
#
#     1    = ciel clair
#     0    = nuage
#     NaN  = terre ou absence de donnée bayésienne

mask_bayes = np.full( (nj, ni), np.nan, dtype=np.float32)

valid_pc = np.isfinite(Pc_2D)

# Clear
# ----------------------------------------------------------

mask_bayes[ ocean & valid_pc & (Pc_2D >= threshold)] = 1.0

# Cloud
# ----------------------------------------------------------

mask_bayes[ ocean & valid_pc & (Pc_2D < threshold)] = 0.0

print("\n======================================")
print("MASQUE BAYESIEN")
print("======================================")
print("Clear :", np.sum(mask_bayes == 1))
print("Cloud :", np.sum(mask_bayes == 0))
print("NaN   :", np.sum(np.isnan(mask_bayes)))

# 12) DIFFERENCE DES MASQUES

# Comparaison uniquement lorsque les deux masques
# sont définis.

valid_comparison = ( np.isfinite(mask_native) & np.isfinite(mask_bayes))

mask_difference = np.full((nj, ni), np.nan, dtype=np.float32)

mask_difference[valid_comparison] = (mask_native[valid_comparison]  -  mask_bayes[valid_comparison])

# 13) STATISTIQUES

n_comparison = np.sum(valid_comparison)

n_agreement = np.sum(
    valid_comparison
    & (mask_difference == 0)
)

n_false_clear = np.sum(
    valid_comparison
    & (mask_difference == 1)
)

n_false_cloud = np.sum(
    valid_comparison
    & (mask_difference == -1)
)


if n_comparison > 0:

    pct_agreement = (
        100.0
        * n_agreement
        / n_comparison
    )

    pct_false_clear = (
        100.0
        * n_false_clear
        / n_comparison
    )

    pct_false_cloud = (
        100.0
        * n_false_cloud
        / n_comparison
    )

else:

    pct_agreement = np.nan
    pct_false_clear = np.nan
    pct_false_cloud = np.nan


print("\n======================================")
print("STATISTIQUES DE COMPARAISON")
print("======================================")

print(
    "Pixels comparés :",
    n_comparison
)

print(
    f"Concordance             : "
    f"{n_agreement:8d} pixels "
    f"({pct_agreement:.2f} %)"
)

print(
    f"Natif clair / Bayes cloud : "
    f"{n_false_clear:8d} pixels "
    f"({pct_false_clear:.2f} %)"
)

print(
    f"Natif cloud / Bayes clair : "
    f"{n_false_cloud:8d} pixels "
    f"({pct_false_cloud:.2f} %)"
)

print(
    "\nSomme des pourcentages :",
    pct_agreement
    + pct_false_clear
    + pct_false_cloud,
    "%"
)

# 14) BT11 AVEC MASQUE NATIF

BT11_native = BT11.copy()

BT11_native[
    mask_native != 1
] = np.nan


# 15) BT11 AVEC MASQUE BAYESIEN

BT11_bayes = BT11.copy()

BT11_bayes[
    mask_bayes != 1
] = np.nan


# 16) COLORMAP TERRE

# Terre affichée en gris/brun sombre
land_cmap = ListedColormap([
    "#4A4A4A"
])

# Le NaN est transparent
land_cmap.set_bad(alpha=0)

mask_cmap = plt.cm.get_cmap("RdYlBu", 2)

# 17) FONCTION DE TRACE BT11

def plot_bt11(
    bt,
    filename,
    title=None
):

    fig, ax = plt.subplots(
        figsize=(9, 7)
    )

    # BT11
    # ------------------------------------------------------

    pcm = ax.pcolormesh(
        lon,
        lat,
        bt,
        cmap="viridis",
        vmin=272,
        vmax=284,
        shading="auto"
    )

    # TERRE
    # ------------------------------------------------------

    ax.pcolormesh(
        lon,
        lat,
        land_hold_on,
        cmap=land_cmap,
        vmin=0,
        vmax=1,
        shading="auto",
        alpha=0.9
    )

    # COLORBAR
    # ------------------------------------------------------

    cbar = fig.colorbar(
        pcm,
        ax=ax
    )

    cbar.set_label(
        "BT11 (K)"
    )

    ax.set_xlabel(
        "Longitude"
    )

    ax.set_ylabel(
        "Latitude"
    )

    if title is not None:
        ax.set_title(title)

    plt.tight_layout()

    outfile = os.path.join(
        outdir,
        filename
    )

    plt.savefig(
        outfile,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close()

    print(
        "Figure sauvegardée :",
        outfile
    )


Lecture Landsat...
Lecture Bayesian...

GRILLE LANDSAT
nj = 8091
ni = 8011
BT11 = (8091, 8011)

MASQUE TERRE
Océan : 52567072
Terre : 12249929

MASQUE NUAGE NATIF
Nuages : 10313012
Clairs : 54503989

MASQUE NATIF FINAL
Clear : 21772451
Cloud : 9069363
NaN   : 33975187

PRODUIT BAYESIEN
Nombre de pixels : 30825984
row : 23 8063
col : 84 7696

Pixels row/col valides : 30825984 / 30825984

CORRESPONDANCE ROW/COL
Distance moyenne : 0.0
Distance médiane : 0.0
Distance maximale : 0.0

Même grille ? True

>>> Utilisation directe de row/col

MASQUE BAYESIEN
Clear : 22232523
Cloud : 8571909
NaN   : 34012569

STATISTIQUES DE COMPARAISON
Pixels comparés : 30804432
Concordance             : 25494961 pixels (82.76 %)
Natif clair / Bayes cloud :  2408313 pixels (7.82 %)
Natif cloud / Bayes clair :  2901158 pixels (9.42 %)

Somme des pourcentages : 100.00000000000001 %


/tmp/ipykernel_19167/1199202779.py:624: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  mask_cmap = plt.cm.get_cmap("RdYlBu", 2)


In [3]:
np.unique(mask_cloud_native)

array([False,  True])

In [4]:
np.unique(cloud_mask_values)

array([0, 1, 2, 3])

In [3]:
ds

<xarray.Dataset> Size: 4GB
Dimensions:                    (nj: 8091, ni: 8011, time: 1)
Coordinates:
  * ni                         (ni) float64 64kB 2.499e+05 ... 4.902e+05
  * nj                         (nj) float64 65kB 5.692e+06 ... 5.45e+06
  * time                       (time) datetime64[ns] 8B 2024-01-18T11:11:28
    lat                        (nj, ni) float32 259MB ...
    lon                        (nj, ni) float32 259MB ...
Data variables:
    satellite_zenith_angle     (nj, ni) float64 519MB ...
    solar_zenith_angle         (nj, ni) float64 519MB ...
    satellite_azimuth_angle    (nj, ni) float64 519MB ...
    solar_azimuth_angle        (nj, ni) float64 519MB ...
    provider_quality_flag      (nj, ni) float64 519MB ...
    dtime                      (nj, ni) float32 259MB ...
    transverse_mercator        int64 8B ...
    brightness_temperature_11  (nj, ni) float32 259MB ...
    brightness_temperature_12  (nj, ni) float32 259MB ...
    cloud_mask                 (nj, ni) float64 519MB 0.0 0.0 0.0 ... 0.0 0.0
    validity_flags             (nj, ni) uint8 65MB ...
Attributes: (12/13)
    summary:                 Landsat-9 L1C radiances compliant with CALISTA p...
    cloud_cover_percentage:  10.04
    geospatial_lon_min:      -6.5899
    geospatial_lon_max:      -3.1345
    geospatial_lat_min:      49.1473
    geospatial_lat_max:      51.381
    ...                      ...
    time_coverage_end:       2024-01-18T11:13:31
    day_night_flag:          Day
    product_version:         1.1
    source:                  CALISTA L1C v1.1
    input_file:              LC09_L1TP_204025_20240118_20240118_02_T1
    history:                 created on 2026-08-13T15:46:56.379984+00:00 from...

In [2]:
# ==========================================================
# 18) CARTE MASQUE NATIF
# ==========================================================

fig, ax = plt.subplots(figsize=(9, 7))

im = ax.pcolormesh(lon, lat, mask_native, cmap=mask_cmap, vmin=0, vmax=1, shading="auto")

# Terre
ax.pcolormesh(lon, lat, land_hold_on, cmap=land_cmap, vmin=0, vmax=1, shading="auto", alpha=0.2)

cb = fig.colorbar(im, ax=ax, ticks=[0.25, 0.75])

cb.ax.set_yticklabels([
    "Cloud",
    "Clear"
])

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.tight_layout()

outfile = os.path.join(
    outdir,
    cfname[0:38] + "native_cloud_mask.png"
)

plt.savefig(
    outfile,
    dpi=150,
    bbox_inches="tight"
)

plt.close()

print(
    "Figure sauvegardée :",
    outfile
)

Figure sauvegardée : /home/kcorrea/Trishna/figures/new3/CALISTA_LC09_L1C_204025_20240118_20240native_cloud_mask.png


In [2]:

# ==========================================================
# 19) CARTE MASQUE BAYESIEN
# ==========================================================

fig, ax = plt.subplots(
    figsize=(9, 7)
)

im = ax.pcolormesh(
    lon,
    lat,
    mask_bayes,
    cmap=mask_cmap,
    vmin=0,
    vmax=1,
    shading="auto"
)

# Terre
ax.pcolormesh(
    lon,
    lat,
    land_hold_on,
    cmap=land_cmap,
    vmin=0,
    vmax=1,
    shading="auto",
    alpha=0.2
)

cb = fig.colorbar(
    im,
    ax=ax,
    ticks=[0.25, 0.75]
)

cb.ax.set_yticklabels([
    "Cloud",
    "Clear"
])

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.tight_layout()

outfile = os.path.join(
    outdir,
    cfname[0:38] + "bayesian_cloud_mask.png"
)

plt.savefig(
    outfile,
    dpi=150,
    bbox_inches="tight"
)

plt.close()

print(
    "Figure sauvegardée :",
    outfile
)


Figure sauvegardée : /home/kcorrea/Trishna/figures/new3/CALISTA_LC09_L1C_204025_20240118_20240bayesian_cloud_mask.png


In [2]:
# ==========================================================
# 20) CARTE DIFFERENCE NATIF - BAYESIEN
# ==========================================================

fig, ax = plt.subplots(figsize=(9, 7))

# Colormap :
#   -1 = natif cloud / bayes clear
#    0 = concordance
#   +1 = natif clear / bayes cloud

diff_cmap = ListedColormap([
    "#2166AC",   # -1
    "#F7F7F7",   #  0
    "#B2182B"    # +1
])

diff_norm = BoundaryNorm(
    [-1.5, -0.5, 0.5, 1.5],
    diff_cmap.N
)

im = ax.pcolormesh(
    lon,
    lat,
    mask_difference,
    cmap=diff_cmap,
    norm=diff_norm,
    shading="auto"
)

# Terre
ax.pcolormesh(
    lon,
    lat,
    land_hold_on,
    cmap=land_cmap,
    vmin=0,
    vmax=1,
    shading="auto",
    alpha=0.9
)

cb = fig.colorbar(
    im,
    ax=ax,
    ticks=[-1, 0, 1]
)

cb.ax.set_yticklabels([
    "Native Cloud / Bayes Clear",
    "Agreement",
    "Native Clear / Bayes Cloud"
])

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.tight_layout()

outfile = os.path.join(
    outdir,
    cfname[0:38] + "native_minus_bayesian.png"
)

plt.savefig(
    outfile,
    dpi=150,
    bbox_inches="tight"
)

plt.close()

print(
    "Figure sauvegardée :",
    outfile
)

Figure sauvegardée : /home/kcorrea/Trishna/figures/new3/CALISTA_LC09_L1C_204025_20240118_20240native_minus_bayesian.png


In [2]:
# ==========================================================
# 21) BT11 AVEC MASQUE NATIF
# ==========================================================

plot_bt11( BT11_native, cfname[0:38] + "BT11_native_clear.png", title=None)

Figure sauvegardée : /home/kcorrea/Trishna/figures/new3/CALISTA_LC09_L1C_204025_20240118_20240BT11_native_clear.png


In [2]:

# ==========================================================
# 22) BT11 AVEC MASQUE BAYESIEN
# ==========================================================

plot_bt11(BT11_bayes, cfname[0:38] + "BT11_bayesian_clear_s01.png", title=None)


# ==========================================================
# 23) FIN
# ==========================================================

print("\n======================================")
print("TRAITEMENT TERMINE")
print("======================================")

Figure sauvegardée : /home/kcorrea/Trishna/figures/new3/CALISTA_LC09_L1C_204025_20240118_20240BT11_bayesian_clear_s01.png

TRAITEMENT TERMINE
